In [13]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import krippendorff

In [ ]:
### CONSTANTS and function definitions ### 
DIMENSIONS = [
    "faithfulness",
    "coverage",
    "retrieval",
    "conciseness",
    "clarity",
]

PRETTY_NAMES = {
    "faithfulness": "Faithfulness",
    "coverage": "Functional Coverage",
    "retrieval": "Retrieval Utility",
    "conciseness": "Conciseness",
    "clarity": "Clarity",
}

JUDGE_DIRS = {
    "GPT-oss-120B": "eval_gpt-oss-120b",
    "Qwen2.5-32B": "eval_qwen2.5-32b-instruct",
    "QwQ-32B": "eval_qwq-32b",
}

def pairwise_absolute_deviation(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    if len(values) < 2:
        return np.nan

    diffs = [abs(a - b) for a, b in combinations(values, 2)]
    return np.mean(diffs)

def load_judge_ratings(root_dir):
    root_dir = Path(root_dir)
    dfs = []

    for judge_name, judge_subdir in JUDGE_DIRS.items():
        judge_dir = root_dir / judge_subdir
        score_files = sorted(judge_dir.glob("*_judge_scores.csv"))

        print(f"{judge_name}: {len(score_files)} files")

        for score_file in score_files:
            df = pd.read_csv(score_file)

            experiment = score_file.name.removesuffix("_judge_scores.csv")

            df["sample_idx"] = np.arange(len(df))
            df["judge"] = judge_name
            df["experiment"] = experiment

            df["item_id"] = (
                experiment
                + "::"
                + df["sample_idx"].astype(str)
            )

            dfs.append(
                df[
                    [
                        "item_id",
                        "experiment",
                        "sample_idx",
                        "cve_id",
                        "judge",
                    ]
                    + DIMENSIONS
                ]
            )

    if not dfs:
        raise ValueError(f"No judge score files found under {root_dir}")

    return pd.concat(dfs, ignore_index=True)

def compute_krippendorff_alpha(ratings):
    rows = []

    for dim in DIMENSIONS:
        matrix = ratings.pivot(
            index="judge",
            columns="item_id",
            values=dim,
        )

        alpha = krippendorff.alpha(
            reliability_data=matrix.to_numpy(dtype=float),
            level_of_measurement="ordinal",
        )

        rows.append({
            "dimension": dim,
            "krippendorff_alpha": alpha,
            "n_items": matrix.shape[1],
        })

    return pd.DataFrame(rows)

def compute_mad(ratings):
    rows = []

    for dim in DIMENSIONS:
        item_mad = (
            ratings
            .groupby("item_id")[dim]
            .apply(pairwise_absolute_deviation)
        )

        rows.append({
            "dimension": dim,
            "mean_pairwise_abs_dev": item_mad.mean(),
            "median_pairwise_abs_dev": item_mad.median(),
            "n_items": item_mad.notna().sum(),
        })

    return pd.DataFrame(rows)

def build_agreement_table(alpha_df, mad_df):
    agreement = alpha_df.merge(
        mad_df[
            [
                "dimension",
                "mean_pairwise_abs_dev",
                "median_pairwise_abs_dev",
            ]
        ],
        on="dimension",
    )

    agreement["Dimension"] = agreement["dimension"].map(PRETTY_NAMES)

    agreement = agreement[
        [
            "Dimension",
            "krippendorff_alpha",
            "mean_pairwise_abs_dev",
            "median_pairwise_abs_dev",
            "n_items",
        ]
    ].rename(columns={
        "krippendorff_alpha": "Krippendorff alpha",
        "mean_pairwise_abs_dev": "Mean MAD",
        "median_pairwise_abs_dev": "Median MAD",
        "n_items": "N",
    })

    return agreement.round(3)

def analyze_judge_agreement(root_dir):
    ratings = load_judge_ratings(root_dir)

    alpha_df = compute_krippendorff_alpha(ratings)
    mad_df = compute_mad(ratings)

    agreement = build_agreement_table(alpha_df, mad_df)

    return agreement

def compute_mean_scores(df):
    df = df.copy()

    df["dataset"] = np.select(
        [
            df["model"].str.contains("pairvul", case=False, na=False),
            df["model"].str.contains("primevul", case=False, na=False),
        ],
        [
            "PairVul",
            "PrimeVul",
        ],
        default="System",
    )

    mean_scores = (
        df.groupby("dataset")[DIMENSIONS]
        .mean()
        .rename(columns=PRETTY_NAMES)
        .round(2)
    )

    return mean_scores

def load_and_compute_mean_scores(filepath):
    df = pd.read_csv(filepath)
    return compute_mean_scores(df)

### **Vul-RAG**

In [16]:
vulrag_agreement = analyze_judge_agreement("vul-rag")
display(vulrag_agreement)

GPT-oss-120B: 44 files
Qwen2.5-32B: 44 files
QwQ-32B: 44 files


,Dimension,Krippendorff alpha,Mean MAD,Median MAD,N
0,Faithfulness,0.191,0.810,0.667,2021
1,Functional Coverage,0.243,0.688,0.667,2021
2,Retrieval Utility,0.192,0.839,0.667,2021
3,Conciseness,0.163,0.785,0.667,2021
4,Clarity,0.381,0.481,0.000,2021


In [23]:
filepath = "vul-rag/eval_gpt-oss-120b/vulrag_eval_gpt-oss-120b_summary.csv"
# filepath = "vul-rag/eval_qwen2.5-32b-instruct/vulrag_eval_qwen2.5-32b-instruct_summary.csv"
# filepath = "vul-rag/eval_qwq-32b/vulrag_eval_qwq-32b_summary.csv"

mean_scores = load_and_compute_mean_scores(filepath)
mean_scores

,Faithfulness,Functional Coverage,Retrieval Utility,Conciseness,Clarity
dataset,,,,,
PairVul,4.07,3.98,4.12,3.73,4.39
PrimeVul,4.12,3.97,4.15,3.78,4.41


### **LLM4Vuln**

In [17]:
llm4vuln_agreement = analyze_judge_agreement("llm4vuln")
display(llm4vuln_agreement)

GPT-oss-120B: 40 files
Qwen2.5-32B: 40 files
QwQ-32B: 40 files


,Dimension,Krippendorff alpha,Mean MAD,Median MAD,N
0,Faithfulness,0.423,0.986,0.667,1840
1,Functional Coverage,0.629,0.726,0.667,1840
2,Retrieval Utility,0.592,0.766,0.667,1840
3,Conciseness,0.279,0.816,0.667,1840
4,Clarity,0.362,0.587,0.667,1840


In [29]:
filepath = "llm4vuln/eval_gpt-oss-120b/llm4vuln_eval_gpt-oss-120b_summary.csv"
# filepath = "llm4vuln/eval_qwen2.5-32b-instruct/llm4vuln_eval_qwen2.5-32b-instruct_summary.csv"
# filepath = "llm4vuln/eval_qwq-32b/llm4vuln_eval_qwq-32b_summary.csv"

mean_scores = load_and_compute_mean_scores(filepath)
mean_scores

,Faithfulness,Functional Coverage,Retrieval Utility,Conciseness,Clarity
dataset,,,,,
PrimeVul,3.73,2.88,3.09,4.03,4.43
System,3.50,2.91,3.16,4.10,4.47
